# 171Yb Rydberg pair C3/C6 with rydcalc

This notebook follows the compact style of `rydcalc/tutorial.ipynb`: choose a Rydberg state, build an `analysis_pair_interaction` object, run a distance scan, and read out fitted `C6` and `C3` coefficients.

The example state follows the anomalous Förster resonance discussed in Peper et al., Phys. Rev. X 15, 011009 (2025):

$$|r\rangle = |\nu=54.56,L=0,F=3/2,m_F=+3/2\rangle.$$


## What C3 and C6 mean

The microscopic Rydberg pair interaction comes from electric dipole-dipole coupling, whose matrix elements scale as `1/R^3`. Near a resonant or nearly resonant pair channel this gives an effective `C3/R^3` interaction.

When the coupled pair channels are far detuned, the same dipole-dipole coupling gives a second-order van der Waals shift, conventionally written as `C6/R^6`.

`rydcalc` fits the tracked pair-branch shift to

$$\Delta E(R)/h = C_6/R^6 + C_3/R^3.$$

Because `rydcalc` uses Hz for energy shifts and micrometers for distance, the fitted units are `C6: Hz um^6` and `C3: Hz um^3`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-rydcalc")

from neutral_yb.external.rydcalc_adapter import load_rydcalc

rydcalc = load_rydcalc()
Yb171 = rydcalc.Ytterbium171(use_db=False)

## Build the pair interaction object

This mirrors the upstream tutorial: `include_opts` is passed to `pair_basis.fill()` through `analysis_pair_interaction`.

In [2]:
st_yb = Yb171.get_state((54.56, 0, 1.5, 1.5), tt="vlfm")

includefn = lambda p, p0: np.abs(p0.energy_Hz - p.energy_Hz) < 10e6
opts = {
    "dn": 2,
    "dl": 2,
    "dm": 1,
    "dipole_allowed": True,
    "pair_include_fn": includefn,
}

pair_int = rydcalc.analysis_pair_interaction(
    st_yb,
    include_opts=opts,
    multipoles=[[1, 1]],
)

print("State:", st_yb)
print("Basis size:", pair_int.pb.dim())

Basis size: 13
State: |171Yb:54.56,L=0,F=1.5,1.5>
Basis size: 13


## Fit C6 and C3

The returned values are `[c6d, c6e, c3d, c3e]`, where `d` means diagonal and `e` means exchange/off-diagonal.

In [3]:
rList_um = np.arange(3, 11, 1)

c6d, c6e, c3d, c3e = pair_int.run(
    rList_um=rList_um,
    th=np.pi / 2,
    Bz_Gauss=5.03,
    silence=True,
)

print(f"C6 diagonal = {c6d:.3e} Hz um^6")
print(f"C3 diagonal = {c3d:.3e} Hz um^3")
print(f"C6 exchange = {c6e:.3e} Hz um^6")
print(f"C3 exchange = {c3e:.3e} Hz um^3")

C6 diagonal = 5.597e+09 Hz um^6
C3 diagonal = 9.340e+08 Hz um^3
C6 exchange = 0.000e+00 Hz um^6
C3 exchange = 0.000e+00 Hz um^3
